# एसआरए निर्णय ट्रांसफार्मर: रूटिंग विश्लेषण

इस डेमो में हम सुदृढीकरण सीखने के लिए एसआरए को **निर्णय ट्रांसफार्मर (डीटी)** के रूप में उपयोग करते हैं, और विश्लेषण करते हैं कि विभिन्न ग्रिडवर्ल्ड (भूलभुलैया) कार्यों (खजाना शिकार, पलायन, आदि) को हल करते समय एजेंट "अपने मस्तिष्क के विभिन्न हिस्सों का उपयोग कैसे करता है"।

हम निम्नलिखित दो बिंदुओं की कल्पना करते हैं:
1. **कार्य पृथक्करण**: पूर्वाग्रह जिसमें सिनैप्स का उपयोग "एस्केप" बनाम "खजाना" के लिए किया जाता है।
2. **धारणा और कार्रवाई का पृथक्करण**: "राज्य", "इनाम", या "कार्रवाई" में दिए गए टोकन प्रकार के आधार पर सिनैप्स प्रभारी कैसे बदलता है।

## 1. पर्यावरण सेटअप

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')


## 2. मॉडल और कार्य तैयार करें
हम 16 विशेषज्ञों (सिनैप्स) के साथ एक मॉडल को परिभाषित करते हैं।

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.sra_language_models import MoESRALanguageModel
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
from src.sra_gridworld import generate_trajectory, make_dt_batch
from src.sra_experiment import make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

config = MoESRAConfig(
    vocab_size=100,
    d_model=64,
    n_layers=2,
    n_heads=2,
    num_synapses=16, # 16 experts ready
    k=2,
    max_seq_len=64
)
model = MoESRALanguageModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=128, max_seq_len=200, pad_idx=0).to(device)
optimizer = make_optimizer(model, lr=0.005)

## 3. मिनी ट्रेनिंग लूप
हम बेतरतीब ढंग से `treasure` और `escape` कार्यों के प्रक्षेप पथ उत्पन्न करते हैं और केवल 50 चरणों के लिए प्रशिक्षण देते हैं। (छोटा रखा गया है ताकि यह कोलाब आदि पर जल्दी खत्म हो जाए)

In [ ]:
print("Training Decision Transformer...")
model.train()

epochs = 150
batch_size = 32
max_steps = 10

for epoch in range(epochs):
    x, y, _ = make_dt_batch(batch_size, max_steps, device)
    
    optimizer.zero_grad()
    outputs, routing_weights = model(x)
    
    # Compute loss only on action prediction
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1), ignore_index=-100)
    
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

print("Training finished!")

## 4. रूटिंग विश्लेषण: प्रति कार्य सिनैप्स उपयोग
प्रशिक्षण के बाद, हम एकत्रित करते हैं और तुलना करते हैं कि राउटर "खजाने की खोज" बनाम "भागने" के लिए कौन से सिनैप्स का उपयोग करना पसंद करता है।

In [ ]:
def analyze_task_usage(task_type, samples=50):
    model.eval()
    usage_counts = torch.zeros(config.num_synapses, device=device)
    
    with torch.no_grad():
        for _ in range(samples):
            traj = generate_trajectory(task_type, max_steps=5)
            x = torch.tensor([traj], dtype=torch.long).to(device)
            _, routing_weights = model(x)
            
            # In the last layer's routing, aggregate the indices of the Synapses that were used
            layer_weights = routing_weights[-1][0]  # [seq_len, n_synapses]
            chosen = layer_weights.argmax(dim=-1)
            usage_counts += torch.bincount(chosen, minlength=config.num_synapses)
            
    usage_pct = (usage_counts / usage_counts.sum()).cpu().numpy()
    return usage_pct

usage_treasure = analyze_task_usage("treasure")
usage_escape = analyze_task_usage("escape")

# Compare with a bar chart
plt.figure(figsize=(10, 5))
x = np.arange(config.num_synapses)
width = 0.35

plt.bar(x - width/2, usage_treasure, width, label='Treasure Task', color='gold')
plt.bar(x + width/2, usage_escape, width, label='Escape Task', color='crimson')

plt.ylabel('Usage Percentage')
plt.xlabel('Synapse Index')
plt.title('Synapse Usage Comparison between Tasks')
plt.xticks(x)
plt.legend()
plt.show()

print("[Observation] Bar heights differ across tasks (i.e. different Synapses are being used).")
print("The router assigns the 'policy for escaping the chaser' and the 'policy for approaching the treasure' to different Synapses.")

## 5. रूटिंग विश्लेषण: धारणा और क्रिया का पृथक्करण (प्रति-टोकन विज़ुअलाइज़ेशन)
एकल "खजाने की खोज" प्रक्षेपवक्र के लिए, हम यह जांचने के लिए हीटमैप का उपयोग करते हैं कि प्रत्येक टोकन (राज्य, इनाम, कार्रवाई) को कौन सा सिनैप्स सौंपा गया था।

In [ ]:
model.eval()
traj = generate_trajectory("treasure", max_steps=2)
x = torch.tensor([traj], dtype=torch.long).to(device)

with torch.no_grad():
    _, routing_weights = model(x)

weights = routing_weights[0][0].cpu().numpy()
from src.constants import ID2TOK
labels = [ID2TOK.get(t, str(t)) for t in traj]

plt.figure(figsize=(12, 8))
sns.heatmap(weights, cmap="YlGnBu", yticklabels=labels, annot=False)
plt.title("Token-wise Synapse Routing (Treasure Task)")
plt.xlabel("Synapse Index")
plt.ylabel("Token Type")
plt.show()

print("[Observation] Looking at the labels on the vertical axis, you can see that for each data type")
print("such as 'State (coordinates)', 'Reward', and 'Action', the highlighted Synapses (horizontal axis) are cleanly separated.")